In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import BayesianRidge
from scipy.stats import pearsonr

In [2]:
def read_data(file_path):
    data = pd.read_csv(file_path, sep='\t', header=0, na_values='nan')
    SNP = data.iloc[:, 4:].apply(pd.to_numeric, errors='coerce').values
    pheno = data.iloc[:, 1].apply(pd.to_numeric, errors='coerce').values
    folds = data.iloc[:, 0].apply(pd.to_numeric, errors='coerce').values
    return SNP, pheno, folds

In [3]:
def bayesian_ridge_regression(X_train, X_test, y_train, y_test):
    model = BayesianRidge()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    pcc, _ = pearsonr(y_test, predictions)
    return pcc, model

In [4]:
def cross_validate_bayesian_ridge(SNP, pheno, folds, num_folds=10):
    pcc_scores = []

    for i in range(1, num_folds + 1):
        test_idx = np.where(folds == i)[0]
        train_idx = np.where(folds != i)[0]

        X_train, X_test = SNP[train_idx], SNP[test_idx]
        y_train, y_test = pheno[train_idx], pheno[test_idx]

        pcc, _ = bayesian_ridge_regression(X_train, X_test, y_train, y_test)
        pcc_scores.append(pcc)

    avg_pcc = np.mean(pcc_scores)
    return avg_pcc

In [5]:
if __name__ == '__main__':
    IMP_input = "IMP_oil.txt"
    QA_input = "QA_oil.txt"

    imp_SNP, imp_pheno, imp_folds = read_data(IMP_input)
    qa_SNP, qa_pheno, qa_folds = read_data(QA_input)

    imp_pcc = cross_validate_bayesian_ridge(imp_SNP, imp_pheno, imp_folds)
    qa_pcc = cross_validate_bayesian_ridge(qa_SNP, qa_pheno, qa_folds)

    print(f"Average PCC (imputed): {imp_pcc:.4f}")
    print(f"Average PCC (non-imputed): {qa_pcc:.4f}")

Average PCC (imputed): 0.4205
Average PCC (non-imputed): 0.6704
